# Task 1

In [5]:
import pandas as pd
import numpy as np
import os
from pandas.testing import assert_frame_equal

# ----------------------------------------------------------------------
# Solution function
def solution(students: pd.DataFrame, subjects: pd.DataFrame, examinations: pd.DataFrame) -> pd.DataFrame:
    """
    Returns for each student and each subject the number of times the student
    attended that exam.
    Ordered by student_id, subject_name.
    """
    # Cross join students and subjects
    cross = students.merge(subjects, how='cross')
    # Count attendance per student+subject
    attendance = (
        examinations.groupby(['student_id', 'subject_name'])
        .size()
        .reset_index(name='attended_exams')
    )
    # Left join to get counts (including 0 for no attendance)
    result = cross.merge(attendance, on=['student_id', 'subject_name'], how='left')
    result['attended_exams'] = result['attended_exams'].fillna(0).astype(int)
    # Order as required
    result = result.sort_values(['student_id', 'subject_name']).reset_index(drop=True)
    # Ensure columns order
    return result[['student_id', 'student_name', 'subject_name', 'attended_exams']]

# ----------------------------------------------------------------------
# Expected computation (independent, using same logic)
def compute_expected(students: pd.DataFrame, subjects: pd.DataFrame, examinations: pd.DataFrame) -> pd.DataFrame:
    if students.empty or subjects.empty:
        return pd.DataFrame(columns=['student_id', 'student_name', 'subject_name', 'attended_exams'])
    cross = students.merge(subjects, how='cross')
    attendance = (
        examinations.groupby(['student_id', 'subject_name'])
        .size()
        .reset_index(name='attended_exams')
    )
    result = cross.merge(attendance, on=['student_id', 'subject_name'], how='left')
    result['attended_exams'] = result['attended_exams'].fillna(0).astype(int)
    result = result.sort_values(['student_id', 'subject_name']).reset_index(drop=True)
    return result[['student_id', 'student_name', 'subject_name', 'attended_exams']]

# ----------------------------------------------------------------------
# Helper to create DataFrames with correct types
def make_students_df(records):
    if records:
        return pd.DataFrame(records)
    else:
        return pd.DataFrame(columns=['student_id', 'student_name']).astype({'student_id': 'int'})

def make_subjects_df(records):
    if records:
        return pd.DataFrame(records)
    else:
        return pd.DataFrame(columns=['subject_name'])

def make_examinations_df(records):
    if records:
        return pd.DataFrame(records)
    else:
        return pd.DataFrame(columns=['student_id', 'subject_name']).astype({'student_id': 'int'})

# ----------------------------------------------------------------------
# Test case generation
test_cases = []

# ----- 1–15: Specific small cases -----

# 1: Basic example from problem
test_cases.append({
    'name': '01_basic_example',
    'students': [
        {'student_id': 1, 'student_name': 'Alice'},
        {'student_id': 2, 'student_name': 'Bob'},
        {'student_id': 13, 'student_name': 'John'},
        {'student_id': 6, 'student_name': 'Alex'},
    ],
    'subjects': [
        {'subject_name': 'Math'},
        {'subject_name': 'Physics'},
        {'subject_name': 'Programming'},
    ],
    'examinations': [
        {'student_id': 1, 'subject_name': 'Math'},
        {'student_id': 1, 'subject_name': 'Physics'},
        {'student_id': 1, 'subject_name': 'Programming'},
        {'student_id': 2, 'subject_name': 'Programming'},
        {'student_id': 1, 'subject_name': 'Physics'},
        {'student_id': 1, 'subject_name': 'Math'},
        {'student_id': 13, 'subject_name': 'Math'},
        {'student_id': 13, 'subject_name': 'Programming'},
        {'student_id': 13, 'subject_name': 'Physics'},
        {'student_id': 2, 'subject_name': 'Math'},
        {'student_id': 1, 'subject_name': 'Math'},
    ]
})

# 2: Single student, single subject, no exams
test_cases.append({
    'name': '02_single_student_single_subject_no_exams',
    'students': [{'student_id': 1, 'student_name': 'Alice'}],
    'subjects': [{'subject_name': 'Math'}],
    'examinations': []
})

# 3: Single student, single subject, one exam
test_cases.append({
    'name': '03_single_student_single_subject_one_exam',
    'students': [{'student_id': 1, 'student_name': 'Alice'}],
    'subjects': [{'subject_name': 'Math'}],
    'examinations': [{'student_id': 1, 'subject_name': 'Math'}]
})

# 4: Single student, single subject, multiple exams
test_cases.append({
    'name': '04_single_student_single_subject_multiple_exams',
    'students': [{'student_id': 1, 'student_name': 'Alice'}],
    'subjects': [{'subject_name': 'Math'}],
    'examinations': [
        {'student_id': 1, 'subject_name': 'Math'},
        {'student_id': 1, 'subject_name': 'Math'},
        {'student_id': 1, 'subject_name': 'Math'}
    ]
})

# 5: Single student, multiple subjects, no exams
test_cases.append({
    'name': '05_single_student_multiple_subjects_no_exams',
    'students': [{'student_id': 1, 'student_name': 'Alice'}],
    'subjects': [
        {'subject_name': 'Math'},
        {'subject_name': 'Physics'},
        {'subject_name': 'Chemistry'}
    ],
    'examinations': []
})

# 6: Single student, multiple subjects, some exams
test_cases.append({
    'name': '06_single_student_multiple_subjects_some_exams',
    'students': [{'student_id': 1, 'student_name': 'Alice'}],
    'subjects': [
        {'subject_name': 'Math'},
        {'subject_name': 'Physics'},
        {'subject_name': 'Chemistry'}
    ],
    'examinations': [
        {'student_id': 1, 'subject_name': 'Math'},
        {'student_id': 1, 'subject_name': 'Math'},
        {'student_id': 1, 'subject_name': 'Physics'},
    ]
})

# 7: Multiple students, single subject, no exams
test_cases.append({
    'name': '07_multiple_students_single_subject_no_exams',
    'students': [
        {'student_id': 1, 'student_name': 'Alice'},
        {'student_id': 2, 'student_name': 'Bob'},
        {'student_id': 3, 'student_name': 'Charlie'},
    ],
    'subjects': [{'subject_name': 'Math'}],
    'examinations': []
})

# 8: Multiple students, single subject, some exams
test_cases.append({
    'name': '08_multiple_students_single_subject_some_exams',
    'students': [
        {'student_id': 1, 'student_name': 'Alice'},
        {'student_id': 2, 'student_name': 'Bob'},
        {'student_id': 3, 'student_name': 'Charlie'},
    ],
    'subjects': [{'subject_name': 'Math'}],
    'examinations': [
        {'student_id': 1, 'subject_name': 'Math'},
        {'student_id': 1, 'subject_name': 'Math'},
        {'student_id': 3, 'subject_name': 'Math'},
    ]
})

# 9: Empty students
test_cases.append({
    'name': '09_empty_students',
    'students': [],
    'subjects': [{'subject_name': 'Math'}],
    'examinations': []
})

# 10: Empty subjects
test_cases.append({
    'name': '10_empty_subjects',
    'students': [{'student_id': 1, 'student_name': 'Alice'}],
    'subjects': [],
    'examinations': []
})

# 11: Both empty
test_cases.append({
    'name': '11_both_empty',
    'students': [],
    'subjects': [],
    'examinations': []
})

# 12: Students with duplicate names? (not a problem, student_id is primary)
test_cases.append({
    'name': '12_duplicate_names',
    'students': [
        {'student_id': 1, 'student_name': 'Alice'},
        {'student_id': 2, 'student_name': 'Alice'},
    ],
    'subjects': [{'subject_name': 'Math'}],
    'examinations': [{'student_id': 1, 'subject_name': 'Math'}]
})

# 13: Examinations with student not in students (should be ignored)
test_cases.append({
    'name': '13_exams_for_nonexistent_student',
    'students': [{'student_id': 1, 'student_name': 'Alice'}],
    'subjects': [{'subject_name': 'Math'}],
    'examinations': [
        {'student_id': 1, 'subject_name': 'Math'},
        {'student_id': 99, 'subject_name': 'Math'},
    ]
})

# 14: Examinations with subject not in subjects (should be ignored)
test_cases.append({
    'name': '14_exams_for_nonexistent_subject',
    'students': [{'student_id': 1, 'student_name': 'Alice'}],
    'subjects': [{'subject_name': 'Math'}],
    'examinations': [
        {'student_id': 1, 'subject_name': 'Math'},
        {'student_id': 1, 'subject_name': 'Physics'},
    ]
})

# 15: Large cross product (few students, many subjects) to test performance
test_cases.append({
    'name': '15_large_cross_small_students_many_subjects',
    'students': [{'student_id': i, 'student_name': f'Student_{i}'} for i in range(1, 4)],
    'subjects': [{'subject_name': f'Subject_{j}'} for j in range(1, 51)],
    'examinations': [
        {'student_id': 1, 'subject_name': f'Subject_{j}'}
        for j in range(1, 51) if j % 2 == 0
    ] + [
        {'student_id': 2, 'subject_name': f'Subject_{j}'}
        for j in range(1, 51) if j % 3 == 0
    ]
})

# ----- 16–35: Small random tests (students up to 5, subjects up to 5, exams up to 30) -----
np.random.seed(42)
for i in range(16, 36):
    n_students = np.random.randint(1, 6)
    n_subjects = np.random.randint(1, 6)
    students = [{'student_id': i, 'student_name': f'Student_{i}'} for i in range(1, n_students+1)]
    subjects = [{'subject_name': f'Subject_{j}'} for j in range(1, n_subjects+1)]
    # Randomly generate exam records: up to n_students * n_subjects * 2
    n_exams = np.random.randint(0, n_students * n_subjects * 2 + 1)
    exams = []
    for _ in range(n_exams):
        sid = np.random.randint(1, n_students+1)
        subj = f'Subject_{np.random.randint(1, n_subjects+1)}'
        exams.append({'student_id': sid, 'subject_name': subj})
    test_cases.append({
        'name': f'16_35_small_random_{i}',
        'students': students,
        'subjects': subjects,
        'examinations': exams
    })

# ----- 36–60: Medium random tests (students up to 50, subjects up to 20, exams up to 200) -----
for i in range(36, 61):
    n_students = np.random.randint(10, 51)
    n_subjects = np.random.randint(5, 21)
    students = [{'student_id': i, 'student_name': f'Student_{i}'} for i in range(1, n_students+1)]
    subjects = [{'subject_name': f'Subject_{j}'} for j in range(1, n_subjects+1)]
    n_exams = np.random.randint(0, n_students * n_subjects * 2 + 1)
    exams = []
    for _ in range(n_exams):
        sid = np.random.randint(1, n_students+1)
        subj = f'Subject_{np.random.randint(1, n_subjects+1)}'
        exams.append({'student_id': sid, 'subject_name': subj})
    test_cases.append({
        'name': f'36_60_medium_random_{i}',
        'students': students,
        'subjects': subjects,
        'examinations': exams
    })

# ----- 61–100: Large random tests (students up to 200, subjects up to 50, exams up to 2000) -----
for i in range(61, 101):
    n_students = np.random.randint(50, 201)
    n_subjects = np.random.randint(20, 51)
    students = [{'student_id': i, 'student_name': f'Student_{i}'} for i in range(1, n_students+1)]
    subjects = [{'subject_name': f'Subject_{j}'} for j in range(1, n_subjects+1)]
    n_exams = np.random.randint(0, min(2000, n_students * n_subjects * 3))
    exams = []
    for _ in range(n_exams):
        sid = np.random.randint(1, n_students+1)
        subj = f'Subject_{np.random.randint(1, n_subjects+1)}'
        exams.append({'student_id': sid, 'subject_name': subj})
    test_cases.append({
        'name': f'61_100_large_random_{i}',
        'students': students,
        'subjects': subjects,
        'examinations': exams
    })

# ----------------------------------------------------------------------
# Run tests
def run_tests(test_dir = os.path.join('.', 'tests_task_1')):
    try:
        os.mkdir(test_dir)
    except:
        pass
    print(f"Running {len(test_cases)} test cases...\n")
    passed = 0
    failed = 0
    i = 1
    for tc in test_cases:
        print(f"Test {i}: {tc['name']} ... ", end='')
        students_df = make_students_df(tc['students'])
        subjects_df = make_subjects_df(tc['subjects'])
        exams_df = make_examinations_df(tc['examinations'])
        expected = compute_expected(students_df, subjects_df, exams_df)
        result = solution(students_df, subjects_df, exams_df)
        try:
            expected.to_csv(os.path.join(test_dir, f'exp_{i}.csv'), index=False)
            expected = pd.read_csv(os.path.join(test_dir, f'exp_{i}.csv'))
            assert_frame_equal(result, expected, check_dtype=False)
            print("PASS")
            passed += 1
        except AssertionError as e:
            print("FAIL")
            failed += 1
            print("  Expected (first 5 rows):")
            print(expected.head(5))
            print("  Got (first 5 rows):")
            print(result.head(5))
            if len(expected) != len(result):
                print(f"  Lengths: expected {len(expected)}, got {len(result)}")
        else:
            students_df.to_csv(os.path.join(test_dir, f'st_{i}.csv'), index=False)
            subjects_df.to_csv(os.path.join(test_dir, f'sub_{i}.csv'), index=False)
            exams_df.to_csv(os.path.join(test_dir, f'ex_{i}.csv'), index=False)
            i += 1


            
    print(f"\nSummary: {passed} passed, {failed} failed out of {len(test_cases)} tests.")

run_tests()

Running 100 test cases...

Test 1: 01_basic_example ... PASS
Test 2: 02_single_student_single_subject_no_exams ... PASS
Test 3: 03_single_student_single_subject_one_exam ... PASS
Test 4: 04_single_student_single_subject_multiple_exams ... PASS
Test 5: 05_single_student_multiple_subjects_no_exams ... PASS
Test 6: 06_single_student_multiple_subjects_some_exams ... PASS
Test 7: 07_multiple_students_single_subject_no_exams ... PASS
Test 8: 08_multiple_students_single_subject_some_exams ... PASS
Test 9: 09_empty_students ... PASS
Test 10: 10_empty_subjects ... PASS
Test 11: 11_both_empty ... PASS
Test 12: 12_duplicate_names ... PASS
Test 13: 13_exams_for_nonexistent_student ... PASS
Test 14: 14_exams_for_nonexistent_subject ... PASS
Test 15: 15_large_cross_small_students_many_subjects ... PASS
Test 16: 16_35_small_random_16 ... PASS
Test 17: 16_35_small_random_17 ... PASS
Test 18: 16_35_small_random_18 ... PASS
Test 19: 16_35_small_random_19 ... PASS
Test 20: 16_35_small_random_20 ... PASS


# Task 2

In [10]:
import os
import pandas as pd
import numpy as np
from pandas.testing import assert_frame_equal

# ============================================================
# ✅ SAMPLE CORRECT SOLUTION (reference implementation)
# ============================================================
def solution(movies: pd.DataFrame, users: pd.DataFrame, movie_rating: pd.DataFrame) -> pd.DataFrame:

    # ---- Part 1: user with most ratings ----
    if not movie_rating.empty and not users.empty:
        user_counts = movie_rating.groupby('user_id').size().reset_index(name='cnt')
        user_counts = user_counts.merge(users, on='user_id', how='inner')

        if not user_counts.empty:
            max_cnt = user_counts['cnt'].max()
            top_user = (
                user_counts[user_counts['cnt'] == max_cnt]
                .sort_values('name')
                .iloc[0]['name']
            )
        else:
            top_user = None
    else:
        top_user = None

    # ---- Part 2: best movie in Feb 2020 ----
    feb = movie_rating[
        (movie_rating['created_at'] >= '2020-02-01') &
        (movie_rating['created_at'] <= '2020-02-29')
    ]

    if not feb.empty and not movies.empty:
        movie_avg = feb.groupby('movie_id')['rating'].mean().reset_index(name='avg')
        movie_avg = movie_avg.merge(movies, on='movie_id', how='inner')

        if not movie_avg.empty:
            max_avg = movie_avg['avg'].max()
            top_movie = (
                movie_avg[movie_avg['avg'] == max_avg]
                .sort_values('title')
                .iloc[0]['title']
            )
        else:
            top_movie = None
    else:
        top_movie = None

    return pd.DataFrame({'results': [top_user, top_movie]})


# ============================================================
# ✅ INDEPENDENT EXPECTED (DIFFERENT STYLE)
# ============================================================
def compute_expected(movies, users, ratings):

    # user
    if len(ratings) > 0:
        counts = ratings.groupby('user_id').size()
        counts = counts.reset_index()
        counts.columns = ['user_id', 'cnt']
        counts = counts.merge(users, on='user_id', how='inner')

        if len(counts) > 0:
            top_user = sorted(
                counts[counts['cnt'] == counts['cnt'].max()]['name']
            )[0]
        else:
            top_user = None
    else:
        top_user = None

    # movie
    feb = ratings[
        (ratings['created_at'] >= '2020-02-01') &
        (ratings['created_at'] <= '2020-02-29')
    ]

    if len(feb) > 0:
        avg = feb.groupby('movie_id')['rating'].mean()
        avg = avg.reset_index()
        avg.columns = ['movie_id', 'avg']
        avg = avg.merge(movies, on='movie_id', how='inner')

        if len(avg) > 0:
            top_movie = sorted(
                avg[avg['avg'] == avg['avg'].max()]['title']
            )[0]
        else:
            top_movie = None
    else:
        top_movie = None

    return pd.DataFrame({'results': [top_user, top_movie]})


# ============================================================
# Helpers
# ============================================================
def make_movies_df(records):
    df = pd.DataFrame(records)
    if df.empty:
        return pd.DataFrame(columns=['movie_id', 'title'])
    return df


def make_users_df(records):
    df = pd.DataFrame(records)
    if df.empty:
        return pd.DataFrame(columns=['user_id', 'name'])
    return df


def make_ratings_df(records):
    df = pd.DataFrame(records)
    if df.empty:
        return pd.DataFrame(columns=['movie_id', 'user_id', 'rating', 'created_at'])
    df['created_at'] = pd.to_datetime(df['created_at'])
    return df


# ============================================================
# ✅ TEST CASE GENERATION
# ============================================================
test_cases = []

# ---- 1–15: deterministic edge cases ----
test_cases.append({
    'name': 'basic',
    'movies': [{'movie_id': 1, 'title': 'A'}],
    'users': [{'user_id': 1, 'name': 'U'}],
    'ratings': [{'movie_id': 1, 'user_id': 1, 'rating': 5, 'created_at': '2020-02-01'}]
})

test_cases.append({
    'name': 'no_ratings',
    'movies': [{'movie_id': 1, 'title': 'A'}],
    'users': [{'user_id': 1, 'name': 'U'}],
    'ratings': []
})

test_cases.append({
    'name': 'user_tie',
    'movies': [{'movie_id': 1, 'title': 'A'}],
    'users': [{'user_id': 1, 'name': 'Bob'}, {'user_id': 2, 'name': 'Alice'}],
    'ratings': [
        {'movie_id': 1, 'user_id': 1, 'rating': 4, 'created_at': '2020-01-01'},
        {'movie_id': 1, 'user_id': 2, 'rating': 5, 'created_at': '2020-01-02'},
    ]
})

test_cases.append({
    'name': 'movie_tie',
    'movies': [{'movie_id': 1, 'title': 'A'}, {'movie_id': 2, 'title': 'B'}],
    'users': [{'user_id': 1, 'name': 'U'}],
    'ratings': [
        {'movie_id': 1, 'user_id': 1, 'rating': 4, 'created_at': '2020-02-01'},
        {'movie_id': 2, 'user_id': 1, 'rating': 4, 'created_at': '2020-02-02'},
    ]
})

test_cases.append({
    'name': 'no_feb',
    'movies': [{'movie_id': 1, 'title': 'A'}],
    'users': [{'user_id': 1, 'name': 'U'}],
    'ratings': [{'movie_id': 1, 'user_id': 1, 'rating': 5, 'created_at': '2020-01-01'}]
})

test_cases.append({
    'name': 'invalid_user',
    'movies': [{'movie_id': 1, 'title': 'A'}],
    'users': [{'user_id': 1, 'name': 'U'}],
    'ratings': [{'movie_id': 1, 'user_id': 2, 'rating': 5, 'created_at': '2020-02-01'}]
})

test_cases.append({
    'name': 'invalid_movie',
    'movies': [{'movie_id': 1, 'title': 'A'}],
    'users': [{'user_id': 1, 'name': 'U'}],
    'ratings': [{'movie_id': 2, 'user_id': 1, 'rating': 5, 'created_at': '2020-02-01'}]
})

test_cases.append({
    'name': 'feb_boundaries',
    'movies': [{'movie_id': 1, 'title': 'A'}],
    'users': [{'user_id': 1, 'name': 'U'}],
    'ratings': [
        {'movie_id': 1, 'user_id': 1, 'rating': 3, 'created_at': '2020-02-01'},
        {'movie_id': 1, 'user_id': 1, 'rating': 5, 'created_at': '2020-02-29'},
    ]
})

# ---- 16–100: randomized ----
np.random.seed(0)

for i in range(16, 101):

    n_movies = np.random.randint(1, 30)
    n_users = np.random.randint(1, 50)
    n_ratings = np.random.randint(0, 500)

    movies = [{'movie_id': i, 'title': f'Movie_{i}'} for i in range(1, n_movies + 1)]
    users = [{'user_id': i, 'name': f'User_{i}'} for i in range(1, n_users + 1)]

    ratings = []
    used_pairs = set()

    for _ in range(n_ratings):
        m = np.random.randint(1, n_movies + 1)
        u = np.random.randint(1, n_users + 1)

        if (m, u) in used_pairs:
            continue
        used_pairs.add((m, u))

        rating = np.random.randint(1, 6)
        month = np.random.choice(
            [1,2,3,4,5,6,7,8,9,10,11,12],
            p=[0.1,0.5,0.1,0.05,0.05,0.03,0.03,0.03,0.03,0.03,0.03,0.02]
        )
        day = np.random.randint(1, 29)

        ratings.append({
            'movie_id': m,
            'user_id': u,
            'rating': rating,
            'created_at': f'2020-{month:02d}-{day:02d}'
        })

    test_cases.append({
        'name': f'random_{i}',
        'movies': movies,
        'users': users,
        'ratings': ratings
    })


# ============================================================
# ✅ RUN TESTS
# ============================================================
def run_tests(test_dir = os.path.join('tests_task_2')):
    try:
        os.mkdir(test_dir)
    except:
        pass
    print(f"Running {len(test_cases)} tests...\n")
    passed, failed = 0, 0

    i = 1
    for tc in test_cases:
        print(f"{i:03d} {tc['name']} ... ", end='')

        movies = make_movies_df(tc['movies'])
        users = make_users_df(tc['users'])
        ratings = make_ratings_df(tc['ratings'])

        expected = compute_expected(movies, users, ratings)
        result = solution(movies, users, ratings)

        try:
            expected.to_csv(os.path.join(test_dir, f'exp_{i}.csv'), index=False)
            expected = pd.read_csv(os.path.join(test_dir, f'exp_{i}.csv'))
            assert_frame_equal(result, expected, check_dtype=False)
            print("PASS")
            passed += 1
        except AssertionError:
            print("FAIL")
            failed += 1
            print("Expected:\n", expected)
            print("Got:\n", result)
            os.remove(os.path.join(test_dir, f'exp_{i}.csv'))
        else:
            movies.to_csv(os.path.join(test_dir, f'mv_{i}.csv'), index=False)
            users.to_csv(os.path.join(test_dir, f'usr_{i}.csv'), index=False)
            ratings.to_csv(os.path.join(test_dir, f'rt_{i}.csv'), index=False)
            i += 1

    print(f"\n✅ Passed: {passed}, ❌ Failed: {failed}")


run_tests()

Running 93 tests...

001 basic ... PASS
002 no_ratings ... PASS
003 user_tie ... PASS
004 movie_tie ... PASS
005 no_feb ... PASS
006 invalid_user ... PASS
007 invalid_movie ... PASS
008 feb_boundaries ... PASS
009 random_16 ... PASS
010 random_17 ... PASS
011 random_18 ... PASS
012 random_19 ... 

C:\Users\Tecno\AppData\Local\Temp\ipykernel_25760\687558055.py:264: FutureWarning: Mismatched null-like values None and nan found. In a future version, pandas equality-testing functions (e.g. assert_frame_equal) will consider these not-matching and raise.
  assert_frame_equal(result, expected, check_dtype=False)
C:\Users\Tecno\AppData\Local\Temp\ipykernel_25760\687558055.py:264: FutureWarning: Mismatched null-like values None and nan found. In a future version, pandas equality-testing functions (e.g. assert_frame_equal) will consider these not-matching and raise.
  assert_frame_equal(result, expected, check_dtype=False)
C:\Users\Tecno\AppData\Local\Temp\ipykernel_25760\687558055.py:264: FutureWarning: Mismatched null-like values None and nan found. In a future version, pandas equality-testing functions (e.g. assert_frame_equal) will consider these not-matching and raise.
  assert_frame_equal(result, expected, check_dtype=False)
C:\Users\Tecno\AppData\Local\Temp\ipykernel_25760\68755805

PASS
013 random_20 ... PASS
014 random_21 ... PASS
015 random_22 ... PASS
016 random_23 ... PASS
017 random_24 ... PASS
018 random_25 ... PASS
019 random_26 ... PASS
020 random_27 ... PASS
021 random_28 ... PASS
022 random_29 ... PASS
023 random_30 ... 

C:\Users\Tecno\AppData\Local\Temp\ipykernel_25760\687558055.py:264: FutureWarning: Mismatched null-like values None and nan found. In a future version, pandas equality-testing functions (e.g. assert_frame_equal) will consider these not-matching and raise.
  assert_frame_equal(result, expected, check_dtype=False)


PASS
024 random_31 ... PASS
025 random_32 ... PASS
026 random_33 ... PASS
027 random_34 ... PASS
028 random_35 ... PASS
029 random_36 ... PASS
030 random_37 ... PASS
031 random_38 ... PASS
032 random_39 ... PASS
033 random_40 ... PASS
034 random_41 ... PASS
035 random_42 ... PASS
036 random_43 ... PASS
037 random_44 ... PASS
038 random_45 ... PASS
039 random_46 ... PASS
040 random_47 ... PASS
041 random_48 ... PASS
042 random_49 ... PASS
043 random_50 ... PASS
044 random_51 ... PASS
045 random_52 ... PASS
046 random_53 ... PASS
047 random_54 ... PASS
048 random_55 ... PASS
049 random_56 ... PASS
050 random_57 ... PASS
051 random_58 ... PASS
052 random_59 ... PASS
053 random_60 ... PASS
054 random_61 ... PASS
055 random_62 ... PASS
056 random_63 ... PASS
057 random_64 ... PASS
058 random_65 ... PASS
059 random_66 ... PASS
060 random_67 ... PASS
061 random_68 ... PASS
062 random_69 ... PASS
063 random_70 ... PASS
064 random_71 ... PASS
065 random_72 ... PASS
066 random_73 ... PASS
067 ra